# Тест API: регистрация, вход, выход, консультации, МКБ-10

Запускайте ячейки **по порядку сверху вниз**.

**Реальные запросы:** бэкенд отправляет код на почту (нужен SMTP в `.env`), после подтверждения в `ACCESS_TOKEN` сохраняется JWT, затем тестируются защищенные эндпоинты `/consultations` и `/mkb10`.

| Сценарий | Что выполнять |
|----------|----------------|
| **Новый email** | шаги `2.1` -> `2.2` (код из письма) |
| **Пользователь уже есть в БД** | только шаг `2.3` |
| **Завершить сессию** | шаг `2.4` |

`BASE_URL`: адрес бэкенда (по умолчанию `http://localhost:8000`). SMTP и JWT-параметры задаются в `.env`.


In [2]:
import json
from datetime import datetime, timezone

import requests

import pandas as pd


In [3]:
BASE_URL = "http://localhost:8000"

ACCESS_TOKEN = None


def print_response(response: requests.Response) -> None:
    print("Status:", response.status_code)
    try:
        data = response.json()
        print("JSON:")
        print(json.dumps(data, ensure_ascii=False, indent=2))
    except Exception:
        print("Text:")
        print(response.text)


def auth_headers() -> dict:
    if not ACCESS_TOKEN:
        return {}
    return {"Authorization": f"Bearer {ACCESS_TOKEN}"}


## 1. Health


In [4]:
print("=== GET /health ===")
health_resp = requests.get(f"{BASE_URL}/health", timeout=10)
print_response(health_resp)


=== GET /health ===
Status: 200
JSON:
{
  "status": "ok"
}


## 2. Учётные данные

Пароль не короче 8 символов. Для **новой** регистрации — email, которого ещё нет в `service_user`.


In [ ]:
TEST_EMAIL = "test@gmail.com"
TEST_PASSWORD = "MySecretPass123"


### 2.1. Новый пользователь — отправить код (`POST /auth/register`)

При `409` email уже занят — используйте другой email или шаг `2.3` (вход). При `503` проверьте SMTP на бэкенде.


In [7]:
print("=== POST /auth/register ===")
register_resp = requests.post(
    f"{BASE_URL}/auth/register",
    json={"login": TEST_EMAIL, "password": TEST_PASSWORD},
    timeout=60,
)
print_response(register_resp)

if register_resp.status_code == 503:
    print("\n>>> Настройте SMTP в .env бэкенда.")
elif register_resp.ok:
    print("\n>>> Проверьте почту, возьмите 4 цифры кода. В следующей ячейке задайте CODE_FROM_EMAIL и запустите её.")


=== POST /auth/register ===
Status: 200
JSON:
{
  "detail": "code_sent"
}

>>> Проверьте почту, возьмите 4 цифры кода. В следующей ячейке задайте CODE_FROM_EMAIL и запустите её.


### 2.2. Подтвердить код (`POST /auth/register/confirm`)

Подставьте код из письма в `CODE_FROM_EMAIL` и выполните ячейку — в `ACCESS_TOKEN` запишется JWT.


In [ ]:
CODE_FROM_EMAIL = "1048"

print("=== POST /auth/register/confirm ===")
confirm_resp = requests.post(
    f"{BASE_URL}/auth/register/confirm",
    json={"login": TEST_EMAIL, "code": CODE_FROM_EMAIL},
    timeout=30,
)
print_response(confirm_resp)

if confirm_resp.ok:
    ACCESS_TOKEN = confirm_resp.json()["access_token"]
    print("\nACCESS_TOKEN сохранён (начало):", ACCESS_TOKEN[:40] + "...")
else:
    print("\nТокен не получен — проверьте код или повторите шаг 3а.")


### 2.3. Вход (`POST /auth/login`) — если пользователь уже в БД

Если только что прошли шаг `2.2`, эту ячейку можно не запускать. Если аккаунт уже создан ранее — выполните **только эту** ячейку вместо шагов `2.1-2.2`.


In [ ]:
print("=== POST /auth/login ===")
login_resp = requests.post(
    f"{BASE_URL}/auth/login",
    json={"login": TEST_EMAIL, "password": TEST_PASSWORD},
    timeout=30,
)
print_response(login_resp)

if login_resp.ok:
    ACCESS_TOKEN = login_resp.json()["access_token"]
    print("\nACCESS_TOKEN сохранён (начало):", ACCESS_TOKEN[:40] + "...")
else:
    print("\nВход не удался — проверьте email/пароль или пройдите регистрацию 3а–3б.")


### 2.4. Выход (`POST /auth/logout`)

Требуется валидный `ACCESS_TOKEN`. После успешного ответа очищаем локальный токен, чтобы завершить сессию в клиенте.

In [8]:
print("=== POST /auth/logout ===")
logout_resp = requests.post(
    f"{BASE_URL}/auth/logout",
    headers=auth_headers(),
    timeout=30,
)
print_response(logout_resp)

if logout_resp.ok:
    ACCESS_TOKEN = None
    print("\nACCESS_TOKEN очищен — пользователь разлогинен.")
else:
    print("\nВыход не выполнен — проверьте, что ACCESS_TOKEN установлен и не истёк.")

=== POST /auth/logout ===
Status: 200
JSON:
{
  "detail": "logged_out"
}

ACCESS_TOKEN очищен — пользователь разлогинен.


## 3. Consultations (нужен непустой `ACCESS_TOKEN`)

Перед запуском этого раздела убедитесь, что выполнен шаг `2.2` или `2.3`. Если уже выполнили `2.4` (выход), сначала снова получите токен через вход.


### 3.1.1. POST

In [ ]:
if not ACCESS_TOKEN:
    raise RuntimeError("Сначала получите токен")

print("=== POST /consultations (create) ===")
create_payload = {
    "consultation_datetime": datetime.now(timezone.utc).isoformat(),
    "complaints": "Боль в груди",
    "disease_anamnesis": "Болеет 3 дня",
    "disease_history": "Перенесён инфаркт миокарда",
    "patient_condition": "Удовлетворительное",
    "life_anamnesis": "Хронические заболевания отрицает",
    "bad_habits": "Курение, алкоголь эпизодически",
    "allergy_anamnesis": "Аллергий не отмечает",
    "vital_parameters": "АД 120/80, ЧСС 70",
    "objective_data": "Физикальный осмотр без особенностей",
    "clinical_assessment": "Стабильное состояние",
    "consultation_conclusion": "Состояние удовлетворительное",
    "main_diagnosis": "I21.0 Острый инфаркт миокарда",
    "comorbid_diagnoses": "E11 Сахарный диабет 2 типа",
    "recommendations": "Наблюдение, ЭКГ, контроль АД",
    "prescriptions": "Аспирин, бета-блокаторы",
    "status_protocol": "draft",
    "protocol_name": "Кардиологический протокол #12",
}
create_resp = requests.post(
    f"{BASE_URL}/consultations",
    json=create_payload,
    headers=auth_headers(),
    timeout=30,
)
print_response(create_resp)

CREATED_CONSULTATION_ID = None
if create_resp.ok:
    CREATED_CONSULTATION_ID = create_resp.json().get("id")
print("CREATED_CONSULTATION_ID:", CREATED_CONSULTATION_ID)


=== POST /consultations (create) ===
Status: 201
JSON:
{
  "external_system": null,
  "external_patient_id": null,
  "external_doctor_id": null,
  "external_visit_id": null,
  "consultation_datetime": "2026-03-31T11:23:10.632828Z",
  "complaints": "Боль в груди",
  "disease_anamnesis": "Болеет 3 дня",
  "disease_history": "Перенесён инфаркт миокарда",
  "patient_condition": "Удовлетворительное",
  "life_anamnesis": "Хронические заболевания отрицает",
  "bad_habits": "Курение, алкоголь эпизодически",
  "allergy_anamnesis": "Аллергий не отмечает",
  "vital_parameters": "АД 120/80, ЧСС 70",
  "objective_data": "Физикальный осмотр без особенностей",
  "clinical_assessment": "Стабильное состояние",
  "consultation_conclusion": "Состояние удовлетворительное",
  "main_diagnosis": "I21.0 Острый инфаркт миокарда",
  "comorbid_diagnoses": "E11 Сахарный диабет 2 типа",
  "recommendations": "Наблюдение, ЭКГ, контроль АД",
  "prescriptions": "Аспирин, бета-блокаторы",
  "protocol_name": "Кардиологи

### 3.1.2. POST (extra_fields)

In [ ]:
if not ACCESS_TOKEN:
    raise RuntimeError("Сначала получите токен")

print("=== POST /consultations (create) ===")
create_payload = {
    "consultation_datetime": datetime.now(timezone.utc).isoformat(),
    "complaints": "test_complaints",
    "protocol_template_id": "ffb5900c-7af3-47af-924f-3c83ef0c0809",
    # "test_field": "test_value",
    # "тестовое_поле": "тестовое_значение",
    "field_name_text": "test_test_text",
    "field_name_number": "987",
    "field_name_date": "2026-01-01",
}

create_resp = requests.post(
    f"{BASE_URL}/consultations",
    json=create_payload,
    headers=auth_headers(),
    timeout=30,
)
print_response(create_resp)

CREATED_CONSULTATION_ID = None
if create_resp.ok:
    CREATED_CONSULTATION_ID = create_resp.json().get("id")
print("CREATED_CONSULTATION_ID:", CREATED_CONSULTATION_ID)


=== POST /consultations (create) ===
Status: 400
JSON:
{
  "detail": "Extra field 'test_field' is not defined in the protocol template"
}
CREATED_CONSULTATION_ID: None


### 3.2.1 GET

In [105]:
if not ACCESS_TOKEN:
    raise RuntimeError("Нет ACCESS_TOKEN")

print("=== GET /consultations (list) ===")
list_resp = requests.get(
    f"{BASE_URL}/consultations",
    headers=auth_headers(),
    timeout=30,
)
print_response(list_resp)


=== GET /consultations (list) ===
Status: 200
JSON:
[
  {
    "id": "b8b17b61-32f9-462a-8706-7c365eeb1d70",
    "protocol_name": null,
    "updated_at": "2026-04-06T07:12:34.785326Z"
  },
  {
    "id": "82cde71f-42ee-417d-b5c5-394cd99e2220",
    "protocol_name": null,
    "updated_at": "2026-04-06T07:07:52.171965Z"
  },
  {
    "id": "01c38f0b-acf3-40c5-bfc9-07b85cf34813",
    "protocol_name": null,
    "updated_at": "2026-04-06T07:00:15.494984Z"
  },
  {
    "id": "92877ec4-b408-47ed-8e3a-b67a650150e5",
    "protocol_name": null,
    "updated_at": "2026-04-06T06:58:20.492396Z"
  },
  {
    "id": "7206f9b8-593f-4700-a045-92add6004406",
    "protocol_name": null,
    "updated_at": "2026-04-06T06:57:12.787157Z"
  },
  {
    "id": "afe9f8ef-7c1c-4849-9a9d-4aa1ba7a2975",
    "protocol_name": null,
    "updated_at": "2026-04-06T06:54:05.910475Z"
  }
]


### 3.2.2. GET {id}

In [111]:
CREATED_CONSULTATION_ID = "b8b17b61-32f9-462a-8706-7c365eeb1d70"
if not CREATED_CONSULTATION_ID:
    raise RuntimeError("Сначала успешный create")

print("=== GET /consultations/{id} ===")
detail_resp = requests.get(
    f"{BASE_URL}/consultations/{CREATED_CONSULTATION_ID}",
    headers=auth_headers(),
    timeout=30,
)
print_response(detail_resp)


=== GET /consultations/{id} ===
Status: 200
JSON:
{
  "external_system": null,
  "external_patient_id": null,
  "external_doctor_id": null,
  "external_visit_id": null,
  "consultation_datetime": "2026-04-06T07:12:34.533196Z",
  "complaints": "test_complaints",
  "disease_anamnesis": null,
  "disease_history": null,
  "patient_condition": null,
  "life_anamnesis": null,
  "bad_habits": null,
  "allergy_anamnesis": null,
  "vital_parameters": null,
  "objective_data": null,
  "clinical_assessment": null,
  "consultation_conclusion": null,
  "main_diagnosis": null,
  "comorbid_diagnoses": null,
  "recommendations": null,
  "prescriptions": null,
  "protocol_name": null,
  "status_protocol": "draft",
  "protocol_template_id": "ffb5900c-7af3-47af-924f-3c83ef0c0809",
  "id": "b8b17b61-32f9-462a-8706-7c365eeb1d70",
  "service_user": "84dfef0a-d42a-48b5-a8ea-0343bf4ea781",
  "extra_fields": {
    "extra_field_1": {
      "type": "text",
      "label": "field_name_text",
      "value": "test_t

### GET {id} view

In [114]:
CREATED_CONSULTATION_ID = "b8b17b61-32f9-462a-8706-7c365eeb1d70"
if not CREATED_CONSULTATION_ID:
    raise RuntimeError("Сначала успешный create")

print("=== GET /consultations/{id}/view ===")
detail_resp = requests.get(
    f"{BASE_URL}/consultations/{CREATED_CONSULTATION_ID}/view",
    headers=auth_headers(),
    timeout=30,
)
print_response(detail_resp)


=== GET /consultations/{id}/view ===
Status: 200
JSON:
{
  "fields": [
    {
      "key": "consultation_datetime",
      "label": "Дата и время консультации",
      "type": "datetime",
      "value": "2026-04-06T07:12:34.533196+00:00",
      "is_extra": false
    },
    {
      "key": "external_system",
      "label": "Идентификатор внешней системы (EMR/МИС)",
      "type": "integer",
      "value": null,
      "is_extra": false
    },
    {
      "key": "external_patient_id",
      "label": "Внешний ID пациента",
      "type": "integer",
      "value": null,
      "is_extra": false
    },
    {
      "key": "external_doctor_id",
      "label": "Внешний ID врача",
      "type": "integer",
      "value": null,
      "is_extra": false
    },
    {
      "key": "external_visit_id",
      "label": "Внешний ID визита/обращения",
      "type": "integer",
      "value": null,
      "is_extra": false
    },
    {
      "key": "complaints",
      "label": "Жалобы",
      "type": "text",
      "

In [120]:
import json

if detail_resp.ok:
    data = detail_resp.json()
    
    with open("detail_resp.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    print("JSON сохранён в detail_resp.json")
else:
    print("Ошибка, файл не сохранён")

JSON сохранён в detail_resp.json


### 3.3.1 PUT

In [ ]:
CREATED_CONSULTATION_ID = "b8b17b61-32f9-462a-8706-7c365eeb1d70"
if not CREATED_CONSULTATION_ID:
    raise RuntimeError("Нет id консультации")

print("=== PUT /consultations/{id} ===")
update_payload = {
    "complaints": "Обновлённые жалобы",
    "recommendations": "Обновлённые рекомендации",
    "status_protocol": "active",
    "protocol_name": "Кардиологический протокол #2",
}
update_resp = requests.put(
    f"{BASE_URL}/consultations/{CREATED_CONSULTATION_ID}",
    json=update_payload,
    headers=auth_headers(),
    timeout=30,
)
print_response(update_resp)


=== PUT /consultations/{id} ===
Status: 200
JSON:
{
  "external_system": null,
  "external_patient_id": null,
  "external_doctor_id": null,
  "external_visit_id": null,
  "consultation_datetime": "2026-04-06T07:12:34.533196Z",
  "complaints": "Обновлённые жалобы",
  "disease_anamnesis": null,
  "disease_history": null,
  "patient_condition": null,
  "life_anamnesis": null,
  "bad_habits": null,
  "allergy_anamnesis": null,
  "vital_parameters": null,
  "objective_data": null,
  "clinical_assessment": null,
  "consultation_conclusion": null,
  "main_diagnosis": null,
  "comorbid_diagnoses": null,
  "recommendations": "Обновлённые рекомендации",
  "prescriptions": null,
  "protocol_name": "Кардиологический протокол #2",
  "status_protocol": "active",
  "protocol_template_id": "ffb5900c-7af3-47af-924f-3c83ef0c0809",
  "id": "b8b17b61-32f9-462a-8706-7c365eeb1d70",
  "service_user": "84dfef0a-d42a-48b5-a8ea-0343bf4ea781",
  "extra_fields": {
    "extra_field_1": {
      "type": "text",
    

### 3.4.1 PATCH {id} view

In [126]:
CREATED_CONSULTATION_ID = "b8b17b61-32f9-462a-8706-7c365eeb1d70"
if not CREATED_CONSULTATION_ID:
    raise RuntimeError("Нет id консультации")

print("=== PATCH /consultations/{id}/view ===")
update_payload = {
    "fields": [
        {
        "key": "complaints",
        "value": "Обновлённые жалобы _2"
        },
        {
        "key": "extra_field_1",
        "value": "новое значение _2"
        }
    ]
}
update_resp = requests.patch(
    f"{BASE_URL}/consultations/{CREATED_CONSULTATION_ID}/view",
    json=update_payload,
    headers=auth_headers(),
    timeout=30,
)
print_response(update_resp)


=== PATCH /consultations/{id}/view ===
Status: 200
JSON:
{
  "fields": [
    {
      "key": "consultation_datetime",
      "label": "Дата и время консультации",
      "type": "datetime",
      "value": "2026-04-06T07:12:34.533196+00:00",
      "is_extra": false
    },
    {
      "key": "external_system",
      "label": "Идентификатор внешней системы (EMR/МИС)",
      "type": "integer",
      "value": null,
      "is_extra": false
    },
    {
      "key": "external_patient_id",
      "label": "Внешний ID пациента",
      "type": "integer",
      "value": null,
      "is_extra": false
    },
    {
      "key": "external_doctor_id",
      "label": "Внешний ID врача",
      "type": "integer",
      "value": null,
      "is_extra": false
    },
    {
      "key": "external_visit_id",
      "label": "Внешний ID визита/обращения",
      "type": "integer",
      "value": null,
      "is_extra": false
    },
    {
      "key": "complaints",
      "label": "Жалобы",
      "type": "text",
     

### 3.5.1 DELETE

In [56]:
if not CREATED_CONSULTATION_ID:
    raise RuntimeError("Нет id консультации")

print("=== DELETE /consultations/{id} ===")
delete_resp = requests.delete(
    f"{BASE_URL}/consultations/{CREATED_CONSULTATION_ID}",
    headers=auth_headers(),
    timeout=30,
)
print_response(delete_resp)


=== DELETE /consultations/{id} ===
Status: 204
Text:



## 4. MKB10 (GET /mkb10) — проверка пагинации и поиска


In [202]:

print("=== GET /mkb10 (searchValue='холер') ===")

resp = requests.get(
    f"{BASE_URL}/mkb10",
    params={"page": 1, "size": 10, "searchValue": "A32"},
    headers=auth_headers(),
    timeout=30,
)
print_response(resp)

data = resp.json()




=== GET /mkb10 (searchValue='холер') ===
Status: 200
JSON:
{
  "page": 1,
  "size": 10,
  "totalCount": 6,
  "items": [
    {
      "code": "A32",
      "name": "Листериоз"
    },
    {
      "code": "A32.0",
      "name": "Кожный листериоз"
    },
    {
      "code": "A32.1",
      "name": "Листериозный менингит и менингоэнцефалит"
    },
    {
      "code": "A32.7",
      "name": "Листериозный сепсис"
    },
    {
      "code": "A32.8",
      "name": "Другие формы листериоза"
    },
    {
      "code": "A32.9",
      "name": "Листериоз неуточненный"
    }
  ]
}


In [35]:
print("=== GET /mkb10 (searchValue='холер') ===")

resp = requests.get(
    f"{BASE_URL}/mkb10",
    params={"page": 1, "size": 10, "searchValue": "холер"},
    headers=auth_headers(),
    timeout=30,
)

data = resp.json()
df = pd.DataFrame(data['items'])
print(f'page = {data['page']} \nsize = {data['size']} \ntotalCount = {data['totalCount']}')
df


=== GET /mkb10 (searchValue='холер') ===
page = 1 
size = 10 
totalCount = 7


,code,name
0,A00,Холера
1,A00.0,"Холера, вызванная холерным вибрионом 01, биова..."
2,A00.1,"Холера, вызванная холерным вибрионом 01, биова..."
3,A00.9,Холера неуточненная
4,Y58.2,Холерная вакцина
5,Z23.0,Необходимость иммунизации только против холеры
6,Z27.0,Необходимость иммунизации против холеры и брюш...


In [36]:
print("=== GET /mkb10 (searchValue='') ===")

resp = requests.get(
    f"{BASE_URL}/mkb10",
    params={"page": 2, "size": 12, "searchValue": ""},
    headers=auth_headers(),
    timeout=30,
)

data = resp.json()
df = pd.DataFrame(data['items'])
print(f'page = {data['page']} \nsize = {data['size']} \ntotalCount = {data['totalCount']}')
df


=== GET /mkb10 (searchValue='') ===
page = 2 
size = 12 
totalCount = 15038


,code,name
0,A02.0,Сальмонеллезный энтерит
1,A02.1,Сальмонеллезный сепсис
2,A02.2,Локализованная сальмонеллезная инфекция
3,A02.8,Другая уточненная сальмонеллезная инфекция
4,A02.9,Сальмонеллезная инфекция неуточненная
5,A03,Шигеллез
6,A03.0,"Шигеллез, вызванный Shigella dysenteriae"
7,A03.1,"Шигеллез, вызванный Shigella flexneri"
8,A03.2,"Шигеллез, вызванный Shigella boydii"
9,A03.3,"Шигеллез, вызванный Shigella sonnei"


## 5. Группы шаблонов

- Создание группы: `POST /template-groups`.
- Обновление названия группы: `PUT /template-groups/{group_id}`.
- Получение списка групп текущего пользователя: `GET /template-groups`.
- Удаление группы: `DELETE /template-groups/{group_id}` — только своя группа (`creator_id` = текущий пользователь); у всех шаблонов с этим `group_id` поле `group_id` сбрасывается в `NULL`; строка группы удаляется из БД.

Ожидания:
- `POST` с новым `group_name` -> `201`.
- `POST` с уже существующим названием у того же пользователя -> `409`.
- `PUT` с новым названием -> `200`, название изменено, `updated_at` обновлен.
- `GET` -> массив объектов с полями `id` и `group_name`.
- `DELETE` -> `204` без тела; чужая или несуществующая группа -> `404`.


In [229]:
GROUP_NAME = "TEST_GROUP_0"

print("=== POST /template-groups (create) ===")
create_group_resp = requests.post(
    f"{BASE_URL}/template-groups",
    json={"group_name": GROUP_NAME},
    headers=auth_headers(),
    timeout=30,
)
print_response(create_group_resp)

=== POST /template-groups (create) ===
Status: 201
JSON:
{
  "id": "5a579a60-ed96-4b34-9196-37c3977c3b67",
  "group_name": "TEST_GROUP_0",
  "creator_id": "69e62db5-653c-4f9b-bd21-05067eef4095",
  "created_at": "2026-04-09T10:39:10.876133Z",
  "updated_at": "2026-04-09T10:39:10.876133Z",
  "deleted_at": null
}


In [76]:
print("=== GET /template-groups ===")
list_groups_resp = requests.get(
    f"{BASE_URL}/template-groups",
    headers=auth_headers(),
    timeout=30,
)
print_response(list_groups_resp)

=== GET /template-groups ===
Status: 200
JSON:
[]


In [43]:
group_data = create_group_resp.json() if create_group_resp.ok else {}
GROUP_ID = group_data.get("id")
OLD_UPDATED_AT = group_data.get("updated_at")
NEW_GROUP_NAME = f"{GROUP_NAME}_updated"

if not GROUP_ID:
    print("GROUP_ID не получен из POST /template-groups. Сначала успешно создайте группу.")
else:
    print("=== PUT /template-groups/{group_id} ===")
    update_group_resp = requests.put(
        f"{BASE_URL}/template-groups/{GROUP_ID}",
        json={"group_name": NEW_GROUP_NAME},
        headers=auth_headers(),
        timeout=30,
    )
    print_response(update_group_resp)

=== PUT /template-groups/{group_id} ===
Status: 200
JSON:
{
  "id": "4cf81dcb-ddd2-4e00-a3be-b3700331b025",
  "group_name": "Кардиология_0_updated",
  "creator_id": "1211b976-d4cd-4e6f-b5a0-bb17fcdd49c5",
  "created_at": "2026-04-01T13:28:16.826601Z",
  "updated_at": "2026-04-01T13:29:07.670355Z",
  "deleted_at": null
}


In [75]:
# _gid = globals().get("GROUP_ID")
_gid = 'eb39aa8b-cbc6-48bc-9c59-e89b71e33d0b'
if not _gid:
    print("GROUP_ID нет — сначала успешно выполните POST /template-groups (ячейка выше).")
else:
    print(f"=== DELETE /template-groups/{{group_id}} (id={_gid}) ===")
    del_group_resp = requests.delete(
        f"{BASE_URL}/template-groups/{_gid}",
        headers=auth_headers(),
        timeout=30,
    )
    print("Status:", del_group_resp.status_code)
    if del_group_resp.status_code == 204:
        print("Группа удалена (204). Шаблоны с этой группой теперь с group_id=null.")
        GROUP_ID = None  # после удаления id больше не существует
    else:
        try:
            print(json.dumps(del_group_resp.json(), ensure_ascii=False, indent=2))
        except Exception:
            print(del_group_resp.text)

=== DELETE /template-groups/{group_id} (id=eb39aa8b-cbc6-48bc-9c59-e89b71e33d0b) ===
Status: 204
Группа удалена (204). Шаблоны с этой группой теперь с group_id=null.


## 6. Шаблоны протоколов

- Список шаблонов: `GET /protocol-templates` — личные шаблоны текущего пользователя (`privacy: personal`, `creator_id` = текущий) и все общие (`privacy: general`). Ответ: массив объектов с полями `id`, `protocol_name`, `group_id`, `privacy`.
- Детали шаблона: `GET /protocol-templates/{template_id}` — те же правила доступа, что у списка; в ответе поля шаблона из `protocol_template` и массив `fields` из `protocol_template_fields`.
- Изменение шаблона: `PUT /protocol-templates/{template_id}` — в теле только те поля, которые нужно изменить (`protocol_name`, `privacy`, `group_id`, `fields`). Личный шаблон может менять только создатель (`creator_id`); общий — только `head_of_department` или `administrator`. При смене/указании `group_id` группа должна быть в списке групп пользователя (`400` при неизвестной ссылке). Установка `privacy: general` — только с подходящей ролью.
- Удаление шаблона: `DELETE /protocol-templates/{template_id}` — физическое удаление строк из `protocol_template_fields` и `protocol_template`. Личный шаблон — только создатель; общий — только `head_of_department` или `administrator`.
- Создание шаблона: `POST /protocol-templates`.

Тело запроса: `protocol_name`, `privacy` (`general` — общий, `personal` — личный), опционально `group_id`, массив `fields` с полями `field_name`, `field_type`, опционально `field_value`.

- `privacy: general` — только для ролей `head_of_department` и `administrator`, иначе `403`.
- `privacy: personal` — без дополнительных проверок роли.
- Если указан `group_id`, группа должна принадлежать текущему пользователю (как в `GET /template-groups`), иначе `400` с текстом про неизвестную ссылку на группу.

Ожидания:
- `GET /protocol-templates` — `200`, массив с полями `id`, `protocol_name`, `group_id`, `privacy`.
- `GET /protocol-templates/{id}` — `200`, полный объект шаблона с ключом `fields`; при отсутствии доступа или записи — `404`.
- `PUT /protocol-templates/{id}` — `200`, обновлённый объект; при правке чужого личного шаблона — `403`; при недостаточных правах на общий — `403`; пустое тело — `400` (`No fields to update`).
- `DELETE /protocol-templates/{id}` — `204` без тела; при удалении чужого личного — `403`; при недостаточных правах на общий — `403`; нет записи — `404`.
- Личный шаблон без группы — `201`, в ответе есть `id`, `fields` с созданными строками.

In [224]:
CONSULTATION_LIKE_FIELDS = [
    {"field_name": "field_name_text", "field_type": "text", "field_value": "test_text"},
    {"field_name": "field_name_number", "field_type": "number", "field_value": "123"},
    {"field_name": "field_name_date", "field_type": "date", "field_value": "2021-01-01"},
]

print("=== POST /protocol-templates ===")
create_tpl_resp = requests.post(
    f"{BASE_URL}/protocol-templates",
    json={
        "protocol_name": "Шаблон как test",
        "privacy": "personal",
        # "privacy": "general",
        "group_id": None,
        # "group_id": "eb39aa8b-cbc6-48bc-9c59-e89b71e33d0b",
        "fields": CONSULTATION_LIKE_FIELDS,
    },
    headers=auth_headers(),
    timeout=30,
)
print_response(create_tpl_resp)

if create_tpl_resp.ok:
    PROTOCOL_TEMPLATE_ID = create_tpl_resp.json().get("id")
    print("Создан шаблон id:", PROTOCOL_TEMPLATE_ID)
else:
    PROTOCOL_TEMPLATE_ID = None

=== POST /protocol-templates ===
Status: 201
JSON:
{
  "id": "c0cbfdcd-cd42-4eeb-8f37-87fa91220e34",
  "protocol_name": "Шаблон как test",
  "group_id": null,
  "creator_id": "69e62db5-653c-4f9b-bd21-05067eef4095",
  "privacy": "personal",
  "created_at": "2026-04-09T10:37:35.030649Z",
  "updated_at": "2026-04-09T10:37:35.030649Z",
  "deleted_at": null,
  "fields": [
    {
      "id": "ff55c687-624b-41c8-b815-67e25af73e0e",
      "field_name": "field_name_text",
      "field_type": "text",
      "field_value": "test_text",
      "created_at": "2026-04-09T10:37:35.030649Z",
      "updated_at": "2026-04-09T10:37:35.030649Z",
      "deleted_at": null
    },
    {
      "id": "6a514f3e-249e-4606-a873-3a8a625609c7",
      "field_name": "field_name_number",
      "field_type": "number",
      "field_value": "123",
      "created_at": "2026-04-09T10:37:35.030649Z",
      "updated_at": "2026-04-09T10:37:35.030649Z",
      "deleted_at": null
    },
    {
      "id": "894f17aa-ec57-47c5-8497-7f7

In [101]:
print("=== GET /protocol-templates ===")
list_tpl_resp = requests.get(
    f"{BASE_URL}/protocol-templates",
    headers=auth_headers(),
    timeout=30,
)
print_response(list_tpl_resp)

=== GET /protocol-templates ===
Status: 200
JSON:
[
  {
    "id": "ffb5900c-7af3-47af-924f-3c83ef0c0809",
    "protocol_name": "Шаблон как test",
    "group_id": null,
    "privacy": "personal"
  },
  {
    "id": "4222543d-b9b4-4edd-ab91-9c1f303eff2b",
    "protocol_name": "Шаблон как test",
    "group_id": "78f370c1-01e3-4436-81e4-c44a5c37e091",
    "privacy": "general"
  },
  {
    "id": "cb24ae4a-608a-4378-a1b4-a6095482ca57",
    "protocol_name": "Шаблон как test",
    "group_id": null,
    "privacy": "general"
  }
]


In [41]:
# _tid = globals().get("PROTOCOL_TEMPLATE_ID")
_tid = '42e5ed09-74f6-40f5-91fc-41c6a8920a6f'
if not _tid:
    print("PROTOCOL_TEMPLATE_ID нет — сначала успешно выполните POST /protocol-templates выше.")
else:
    print(f"=== GET /protocol-templates/{{template_id}} (id={_tid}) ===")
    detail_tpl_resp = requests.get(
        f"{BASE_URL}/protocol-templates/{_tid}",
        headers=auth_headers(),
        timeout=30,
    )
    print_response(detail_tpl_resp)

=== GET /protocol-templates/{template_id} (id=42e5ed09-74f6-40f5-91fc-41c6a8920a6f) ===
Status: 200
JSON:
{
  "id": "42e5ed09-74f6-40f5-91fc-41c6a8920a6f",
  "protocol_name": "Шаблон как test",
  "group_id": null,
  "creator_id": "cb4adf37-5e82-48f7-bc7e-ccc52cac53f1",
  "privacy": "personal",
  "created_at": "2026-04-05T10:46:02.446986Z",
  "updated_at": "2026-04-05T10:46:02.446986Z",
  "deleted_at": null,
  "fields": [
    {
      "id": "07957d9e-05d1-4807-9f22-3facc3386f4e",
      "field_name": "field_name_text",
      "field_type": "text",
      "field_value": "test_text",
      "created_at": "2026-04-05T10:46:02.446986Z",
      "updated_at": "2026-04-05T10:46:02.446986Z",
      "deleted_at": null
    },
    {
      "id": "dc6f6e53-51f3-4929-b313-044e9c0e7311",
      "field_name": "field_name_number",
      "field_type": "number",
      "field_value": "123",
      "created_at": "2026-04-05T10:46:02.446986Z",
      "updated_at": "2026-04-05T10:46:02.446986Z",
      "deleted_at": nul

In [50]:
# _tid = globals().get("PROTOCOL_TEMPLATE_ID")
_tid = '42e5ed09-74f6-40f5-91fc-41c6a8920a6f'
if not _tid:
    print("PROTOCOL_TEMPLATE_ID нет — сначала успешно выполните POST /protocol-templates.")
else:
    print(f"=== PUT /protocol-templates/{{template_id}} (id={_tid}) ===")
    put_tpl_resp = requests.put(
        f"{BASE_URL}/protocol-templates/{_tid}",
        json={
            "protocol_name": "Шаблон как test (обновлён)",
            "group_id": "8aba4872-718d-4e2c-ae04-7dd40a9da728",
            "fields": [
                {"field_name": "field_name_text", "field_type": "text", "field_value": "updated_text"},
                {"field_name": "field_name_number", "field_type": "number", "field_value": "999"},
            ],
        },
        headers=auth_headers(),
        timeout=30,
    )
    print_response(put_tpl_resp)

=== PUT /protocol-templates/{template_id} (id=42e5ed09-74f6-40f5-91fc-41c6a8920a6f) ===
Status: 200
JSON:
{
  "id": "42e5ed09-74f6-40f5-91fc-41c6a8920a6f",
  "protocol_name": "Шаблон как test (обновлён)",
  "group_id": "8aba4872-718d-4e2c-ae04-7dd40a9da728",
  "creator_id": "cb4adf37-5e82-48f7-bc7e-ccc52cac53f1",
  "privacy": "personal",
  "created_at": "2026-04-05T10:46:02.446986Z",
  "updated_at": "2026-04-05T11:21:45.802771Z",
  "deleted_at": null,
  "fields": [
    {
      "id": "012df743-6869-4c49-9885-c68b93df309e",
      "field_name": "field_name_text",
      "field_type": "text",
      "field_value": "updated_text",
      "created_at": "2026-04-05T11:21:45.802771Z",
      "updated_at": "2026-04-05T11:21:45.802771Z",
      "deleted_at": null
    },
    {
      "id": "7b470e8c-6905-434c-b4ce-2329d4b33615",
      "field_name": "field_name_number",
      "field_type": "number",
      "field_value": "999",
      "created_at": "2026-04-05T11:21:45.802771Z",
      "updated_at": "2026-

In [ ]:
_tid = 'c0cbfdcd-cd42-4eeb-8f37-87fa91220e34'

print(f"=== PUT /protocol-templates/{{template_id}} (id={_tid}) ===")
put_tpl_resp = requests.put(
    f"{BASE_URL}/protocol-templates/{_tid}",
    json={
        "group_id": "5a579a60-ed96-4b34-9196-37c3977c3b67",
        # "group_id": None,
    },
    headers=auth_headers(),
    timeout=30,
)
print_response(put_tpl_resp)

=== PUT /protocol-templates/{template_id} (id=c0cbfdcd-cd42-4eeb-8f37-87fa91220e34) ===
Status: 200
JSON:
{
  "id": "c0cbfdcd-cd42-4eeb-8f37-87fa91220e34",
  "protocol_name": "Шаблон как test",
  "group_id": "5a579a60-ed96-4b34-9196-37c3977c3b67",
  "creator_id": "69e62db5-653c-4f9b-bd21-05067eef4095",
  "privacy": "personal",
  "created_at": "2026-04-09T10:37:35.030649Z",
  "updated_at": "2026-04-09T10:39:28.240349Z",
  "deleted_at": null,
  "fields": [
    {
      "id": "ff55c687-624b-41c8-b815-67e25af73e0e",
      "field_name": "field_name_text",
      "field_type": "text",
      "field_value": "test_text",
      "created_at": "2026-04-09T10:37:35.030649Z",
      "updated_at": "2026-04-09T10:37:35.030649Z",
      "deleted_at": null
    },
    {
      "id": "6a514f3e-249e-4606-a873-3a8a625609c7",
      "field_name": "field_name_number",
      "field_type": "number",
      "field_value": "123",
      "created_at": "2026-04-09T10:37:35.030649Z",
      "updated_at": "2026-04-09T10:37:35

In [59]:
# _tid = globals().get("PROTOCOL_TEMPLATE_ID")
_tid = '42e5ed09-74f6-40f5-91fc-41c6a8920a6f'
if not _tid:
    print("PROTOCOL_TEMPLATE_ID нет — сначала успешно выполните POST /protocol-templates.")
else:
    print(f"=== DELETE /protocol-templates/{{template_id}} (id={_tid}) ===")
    del_tpl_resp = requests.delete(
        f"{BASE_URL}/protocol-templates/{_tid}",
        headers=auth_headers(),
        timeout=30,
    )
    print("Status:", del_tpl_resp.status_code)
    if del_tpl_resp.status_code == 204:
        print("Удалено (204 No Content). Повторный GET по этому id вернёт 404.")
    else:
        try:
            print(json.dumps(del_tpl_resp.json(), ensure_ascii=False, indent=2))
        except Exception:
            print(del_tpl_resp.text)

=== DELETE /protocol-templates/{template_id} (id=42e5ed09-74f6-40f5-91fc-41c6a8920a6f) ===
Status: 204
Удалено (204 No Content). Повторный GET по этому id вернёт 404.


## 7. Автодополнение

In [241]:
ML_MKB_FIELD_NAME = "test_field_123"
ML_MKB_DESCRIPTION = "ОРВИ"
ML_MKB_N_CANDIDATES = -1
ML_MKB_USE_LLM = True
ML_AUTOCOMPLETE_PREFIX = "боль в"
ML_AUTOCOMPLETE_SUGGESTIONS_COUNT = 3

LAST_SELECTED_TEXT = ""
LAST_EVENT_TYPE = "form_click"

In [239]:
if not ACCESS_TOKEN:
    print("ACCESS_TOKEN отсутствует — сначала выполните шаги логина/регистрации.")
else:
    print("=== POST /ml_mkb_code ===")
    ml_mkb_resp = requests.post(
        f"{BASE_URL}/ml_mkb_code",
        json={
            "field_name": ML_MKB_FIELD_NAME,
            "description": ML_MKB_DESCRIPTION,
            "n_candidates": ML_MKB_N_CANDIDATES,
            "use_llm": ML_MKB_USE_LLM,
        },
        headers=auth_headers(),
        timeout=60,
    )
    print_response(ml_mkb_resp)

=== POST /ml_mkb_code ===
Status: 200
JSON:
{
  "extracted_diagnosis": "Острая инфекция верхних дыхательных путей",
  "candidates": [
    {
      "code": "J06.9",
      "name": "J06.9 - Острая инфекция верхних дыхательных путей неуточненная",
      "distance": 0.28451400995254517,
      "similarity": 0.7154859900474548
    }
  ],
  "best_code": "J06.9",
  "best_name": "J06.9 - Острая инфекция верхних дыхательных путей неуточненная"
}


In [240]:
if not ACCESS_TOKEN:
    print("ACCESS_TOKEN отсутствует — сначала выполните шаги логина/регистрации.")
else:
    print("=== POST /ml_autocomplete ===")
    ml_mkb_resp = requests.post(
        f"{BASE_URL}/ml_autocomplete",
        json={
            "field_name": ML_MKB_FIELD_NAME,
            "prefix": ML_AUTOCOMPLETE_PREFIX,
            "suggestions_count": ML_AUTOCOMPLETE_SUGGESTIONS_COUNT,
        },
        headers=auth_headers(),
        timeout=60,
    )
    print_response(ml_mkb_resp)

=== POST /ml_autocomplete ===
Status: 200
JSON:
{
  "suggestions": [
    "боль вколенных суста",
    "боль вколенном суста",
    "боль вгорле, нас"
  ]
}


In [242]:
if not ACCESS_TOKEN:
    print("ACCESS_TOKEN отсутствует — сначала выполните шаги логина/регистрации.")
else:
    print("=== POST /ml_autocomplete_stats ===")
    ml_mkb_select_resp = requests.post(
        f"{BASE_URL}/ml_autocomplete_stats",
        json={
            "field_name": ML_MKB_FIELD_NAME,
            "description": ML_MKB_DESCRIPTION,
            "n_candidates": ML_MKB_N_CANDIDATES,
            "selected_text": LAST_SELECTED_TEXT,
            "event_type": LAST_EVENT_TYPE,
        },
        headers=auth_headers(),
        timeout=30,
    )
    print_response(ml_mkb_select_resp)

=== POST /ml_autocomplete_stats ===
Status: 200
JSON:
{
  "detail": "accepted"
}


## 8. Проверка на полноту

In [12]:
# EXAMPLE_PROMPT = "Сильно болит голова и подташнивает."
EXAMPLE_PROMPT = "Боль в поячничной области, с иррадиацией в правую ногу, до 7 баллов. на слабость в ногах, ограничение самостоятельного передвижения Периодически - отеки голеней."

In [13]:
if not ACCESS_TOKEN:
    print("ACCESS_TOKEN отсутствует — сначала выполните шаги логина/регистрации.")
else:
    print("=== POST /ml_fullness_complaints (incomplete case) ===")
    fullness_resp = requests.post(
        f"{BASE_URL}/ml_fullness_complaints",
        json={"example_prompt": EXAMPLE_PROMPT},
        headers=auth_headers(),
        timeout=60,
    )
    print_response(fullness_resp)

=== POST /ml_fullness_complaints (incomplete case) ===
Status: 200
JSON:
{
  "status": "full",
  "color": "green",
  "analysis": "",
  "suggested_text": ""
}


## 9. Service User

In [12]:
if not ACCESS_TOKEN:
    print("ACCESS_TOKEN отсутствует — сначала выполните шаги логина/регистрации.")
else:
    print("=== GET /service_user ===")
    service_user_resp = requests.get(
        f"{BASE_URL}/service_user",
        headers=auth_headers(),
        timeout=30,
    )
    print_response(service_user_resp)

=== GET /service_user ===
Status: 200
JSON:
{
  "login": "test32",
  "first_name": "test1",
  "last_name": "testest2",
  "role": "doctor",
  "is_active": true,
  "created_at": "2026-04-21T13:24:05.365780Z",
  "updated_at": null,
  "deleted_at": null
}
